# Compare two saved diffusion models

Run from top to bottom: list saved runs, select A and B, show shared noise and adjacent sample grids, inspect training progress, then reconstruct real validation images.

Runs are identified by experiment name and a timestamp plus unique ID. `best` and `last` select checkpoints within a run; there is no numbered model-version scheme. Plotting and sampling helpers live in `radio_diffusion.review`.

In [ ]:
import torch
from IPython.display import HTML, display

from radio_diffusion.experiment import load_experiment_checkpoint
from radio_diffusion.review import (
    DEFAULT_OUTPUT_ROOT, check_comparison, common_validation_images, denoise,
    discover_runs, run_table, seeded_noise, show_grids, show_loss_curves,
    show_reconstructions, show_training_progress,
)

# Review settings. Change seed to choose new noise and new validation examples.
seed = 42
n_samples = 16                  # 16 images per model, displayed as 4 × 4
batch_size = 4                  # Keep fixed for reproducible reverse-step randomness.
n_real_images = 4
reconstruction_timestep = 999   # 0 .. num_steps - 1; larger means more corruption.
display_gamma = 0.5             # Display contrast only; never changes model inputs.
display_limits = (-0.2, 5) # None           # Shared (minimum, maximum); None = automatic.
independent_colorbars = True   # True: automatic scale and colorbar for every image.
output_root = DEFAULT_OUTPUT_ROOT  # Override if saved runs are stored elsewhere.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Available models and selection

All checkpointed runs are listed alphabetically by experiment name, newest run first within each name. Choose the two table indices below. `auto` loads the best checkpoint (including a version suffix), falling back to the last checkpoint; an explicit checkpoint filename is also supported, including Lightning suffixes such as `best-v1.ckpt`.

In [ ]:
runs = discover_runs(output_root)
print(f"{len(runs)} saved runs across {len({run.name for run in runs})} experiment names")
display(HTML(run_table(runs)))

In [ ]:
run_indices = (0, 1)                   # Model A, model B
checkpoint_choices = ("auto", "auto")  # Or "best", "last", or a displayed filename.

if len(run_indices) != 2 or any(index < 0 or index >= len(runs) for index in run_indices):
    raise ValueError("Choose two available run indices. One run can be compared using different checkpoints.")
if len(checkpoint_choices) != 2:
    raise ValueError("Choose one checkpoint for A and one for B.")
selected_runs = [runs[index] for index in run_indices]
checkpoints = [run.checkpoint(choice) for run, choice in zip(selected_runs, checkpoint_choices)]
if checkpoints[0].resolve() == checkpoints[1].resolve():
    raise ValueError("Select two different runs or different checkpoints of the same run.")

# Keep models on CPU between sampling calls so they can share a single GPU.
modules = [load_experiment_checkpoint(path, map_location="cpu") for path in checkpoints]
image_shape = check_comparison(selected_runs, modules)
labels = [f"{letter}: {run.name}" for letter, run in zip("AB", selected_runs)]
for label, checkpoint in zip(labels, checkpoints):
    print(f"{label}\n  {checkpoint}")
print(f"Sampling device: {device}; shared image shape: {image_shape}")

## 2. Shared starting noise

These exact Gaussian noise tensors will be fed into **both networks**. Positions are preserved in the result grids: top-left input corresponds to top-left output in A and B. `seed` fixes these inputs independently of cell execution order. White borders separate the individual images in the input and result grids.

In [ ]:
if n_samples < 1:
    raise ValueError("n_samples must be positive.")
initial_noise = seeded_noise((n_samples, *image_shape), seed)
show_grids(
    [initial_noise], [f"Shared inputs for A and B — seed {seed}"], gamma=1,
    independent_colorbars=independent_colorbars,
)

## 3. Generated images: A beside B

Each model denoises the shared inputs through its full reverse process. Both use the same seeded reverse-step randomness. Matching positions in the two 4×4 grids are paired examples.

With `independent_colorbars=False`, the images share one raw-intensity scale. Set `display_limits` to `None` for automatic limits, or to a `(minimum, maximum)` pair for fixed limits across generated grids, training histories, and original/reconstructed images. Lower the maximum to reveal faint structure; values outside the chosen range use the endpoint colors. Noise displays keep their own automatic scales.

With `independent_colorbars=True`, each individual image gets its own colorbar and automatic intensity range. This ignores `display_limits` and applies to all image plots, including inputs and training histories. Colors represent different intensities in different tiles; use the individual colorbar values to compare brightness.

`display_gamma` adjusts visual contrast without changing model inputs. Reproducibility assumes the same device, batch size, and software setup.

In [ ]:
generated_images = [
    denoise(module, initial_noise, seed=seed + 1, device=device, batch_size=batch_size, label=label)
    for module, label in zip(modules, labels)
]

In [ ]:
show_grids(
    generated_images, labels, gamma=display_gamma, limits=display_limits,
    independent_colorbars=independent_colorbars,
)

## 4. Progress across training epochs

The loss plot uses saved CSV logs. Below it, each row follows the same saved sample through increasing epochs, with at most six columns and two sample rows per model. By default, scales are shared across both models' histories. With `independent_colorbars=True`, each saved image has its own automatic range and colorbar.

These are archived images generated with the **training preview seed**, shown in each title. Changing this notebook's seed cannot change archived samples. Current configs save two samples every 25 epochs. Missing or single snapshots are reported explicitly. Intermediate reverse steps were not saved, so historical denoising trajectories cannot be reconstructed from these final images alone.

In [ ]:
show_loss_curves(selected_runs, labels)
show_training_progress(
    selected_runs, labels, gamma=display_gamma, limits=display_limits,
    independent_colorbars=independent_colorbars,
)

## 5. Reconstruct real images

Choose seeded validation images held out from **both** training runs, preserve their original intensity values, and corrupt them once at `reconstruction_timestep`. Feed the identical noisy images into A and B, reversing from that timestep down to zero with shared seeded randomness.

By default, the original and both reconstructions share an intensity scale, and the noisy-input column has a separate labelled scale. With `independent_colorbars=True`, every original, noisy input, and reconstruction has its own automatic scale and colorbar. Larger timesteps remove more information; these unconditional, stochastic models may produce a plausible variation instead of recovering the original exactly.

The dataset must already be available at the saved configs' data paths (or their mounted remote paths). This review reads local data and does not start a remote sync.

In [ ]:
clean_images, image_paths = common_validation_images(selected_runs, count=n_real_images, seed=seed)
if tuple(clean_images.shape[1:]) != image_shape:
    raise ValueError("Validation images do not match the saved sample shape.")
for index, path in enumerate(image_paths, start=1):
    print(f"Image {index}: {path.name}")

noise = seeded_noise(clean_images.shape, seed + 2)
timesteps = torch.full((len(clean_images),), reconstruction_timestep, dtype=torch.long)
noisy_images, _ = modules[0].diffusion.add_noise(clean_images, timesteps, noise=noise)
reconstructions = [
    denoise(module, noisy_images, seed=seed + 3, device=device, batch_size=batch_size,
            start_timestep=reconstruction_timestep, label=f"{label} reconstruction")
    for module, label in zip(modules, labels)
]
show_reconstructions(
    clean_images, noisy_images, reconstructions, labels,
    gamma=display_gamma, limits=display_limits,
    independent_colorbars=independent_colorbars,
)